<div align="center">

# Universidad de San Martín

## Infraestructura para Ciencia de Datos

### Licenciatura en Ciencia de Datos

<img src="../logo.jpg" alt="Logo UNSAM" width="300"/>

---

</div>

# 🏗️ Clase 04: La Refinería (Capa Silver)

---

## 🎯 Objetivo de Hoy
Transformar los datos crudos de la **Capa Bronze** en datos técnicos limpios y confiables (**Capa Silver**). 

En esta clase nos enfocamos en **Fixing** (arreglar errores) y **Shaping** (dar forma) para que los datos sean consumibles por humanos y máquinas.

---

## 🧪 Escenario: Limpieza de Ventas

Vamos a trabajar con un dataset de ventas que tiene múltiples problemas: nulos, duplicados, inconsistencias en categorías y formatos de fecha mezclados.

> 🔁 **Continuidad con clase03 — el mismo contrato, otra capa.**
>
> En clase03 introdujimos el **Data Contract** declarativo en `stack/data/contracts/ventas.yaml` y vimos cómo Bronze valida la **forma** del archivo (extensión, encoding, delimiter, columnas presentes).
>
> Ahora en Silver toca la **semántica**: leer el mismo YAML y aplicar `quality_rules` + `evolution_policy` fila por fila. La sección siguiente lo implementa con **Pydantic**, que es el equivalente Python del contrato YAML — los `Field(gt=0, ...)` mapean directo a las reglas declarativas.
>
> **Bronze pregunta:** *¿este archivo se puede leer?* — **Silver pregunta:** *¿estos datos son confiables?*

### 🛡️ Elevación Técnica: Contratos con Pydantic

**Pydantic** es el estándar de la industria para validar datos a nivel fila (lo usa FastAPI): rápido, declarativo y basado en el **type hinting** nativo de Python. En Silver lo usamos para convertir cada fila de Bronze en un objeto **validado y normalizado**.

> [!TIP]
> Cada `Field(gt=0)`, `min_length=2`, `EmailStr` o el tipo `date` es **una regla declarativa** — el equivalente Python de los `quality_rules` del `ventas.yaml` (clase03). Si la fila cumple, Pydantic devuelve el objeto **normalizado** (ej. `'  lApToP  '` → `'Laptop'`); si no, lanza `ValidationError`. Esa excepción es **exactamente** el mecanismo con el que Silver separa válidos de cuarentena (lo vas a ver en el DAG `silver_02_contrato`).

Probá abajo casos buenos y malos:


In [6]:
from pydantic import BaseModel, Field, field_validator, EmailStr
from typing import Optional
from datetime import date
import pandas as pd



class VentaContract(BaseModel):
    """Contrato de Datos para Silver Ventas (equivalente Python del ventas.yaml)."""
    venta_id: int = Field(gt=0)
    producto: str = Field(min_length=2)
    precio: float = Field(ge=0)
    cantidad: int = Field(default=1, ge=1)
    fecha: date
    cliente_email: Optional[EmailStr] = None

    @field_validator('producto')
    @classmethod
    def normalizar_producto(cls, v: str) -> str:
        return v.strip().title()


casos = [
    ("válido (se normaliza)", {'venta_id': 123, 'producto': '  lApToP  ', 'precio': 1500.0,'cantidad': 2, 'fecha': '2024-05-20', 'cliente_email': 'JUAN@EXAMPLE.COM'}),
    ("precio negativo → viola ge=0", {'venta_id': 124, 'producto': 'Mouse', 'precio': -50.0, 'fecha': '2024-05-20'}),
    ("venta_id = 0 → viola gt=0", {'venta_id': 0, 'producto': 'Teclado', 'precio': 75.0, 'fecha': '2024-05-20'}),
    ("email inválido → viola EmailStr", {'venta_id': 125, 'producto': 'Monitor', 'precio': 350.0,'fecha': '2024-05-20', 'cliente_email': 'no-es-mail'}),
]

for desc, fila in casos:
    try:
        v = VentaContract(**fila)
        print(f"✅ {desc}: OK → {v.model_dump()}")
    except Exception as e:
        print(f"❌ {desc}: RECHAZADO → {type(e).__name__}: {str(e)[:120]}")


✅ válido (se normaliza): OK → {'venta_id': 123, 'producto': 'Laptop', 'precio': 1500.0, 'cantidad': 2, 'fecha': datetime.date(2024, 5, 20), 'cliente_email': 'JUAN@example.com'}
❌ precio negativo → viola ge=0: RECHAZADO → ValidationError: 1 validation error for VentaContract
precio
  Input should be greater than or equal to 0 [type=greater_than_equal, input
❌ venta_id = 0 → viola gt=0: RECHAZADO → ValidationError: 1 validation error for VentaContract
venta_id
  Input should be greater than 0 [type=greater_than, input_value=0, input_
❌ email inválido → viola EmailStr: RECHAZADO → ValidationError: 1 validation error for VentaContract
cliente_email
  value is not a valid email address: An email address must have an @


## 📑 1. Contratos de Datos (Data Quality)

En la ingeniería moderna, no limpiamos datos "a ver qué pasa". Definimos **Contratos**: expectativas claras que el dato debe cumplir para pasar de Bronze a Silver.

### 🎯 Principios de un Contrato de Datos

Un contrato de datos define:
1. **Schema**: Qué columnas esperar y sus tipos
2. **Constraints**: Reglas que los datos DEBEN cumplir
3. **Actions**: Qué hacer cuando se violan las reglas

### 📋 Tabla de Contratos - Dataset Ventas

| Campo | Expectativa | Acción en caso de Fallo | Severidad | ¿Por qué? |
| :--- | :--- | :--- | :--- | :--- |
| `venta_id` | NOT NULL, UNIQUE | **Quarantine** | 🔴 CRITICAL | Sin ID no hay trazabilidad ni idempotencia. |
| `fecha` | Formato ISO 8601 | **Hard Fail** | 🔴 CRITICAL | Fechas mal formateadas rompen el linaje temporal. |
| `monto` | > 0, tipo FLOAT | **Log & Fix** | 🟡 MEDIUM | Montos negativos o NULL suelen ser errores de carga. |
| `cliente_id` | Existe en `dim_clientes` | **Log & Continue** | 🟢 LOW | Podemos crear cliente "Desconocido" temporalmente. |
| `producto` | En catálogo válido | **Auto-correct** | 🟡 MEDIUM | Typos comunes: "Laptos" → "Laptop" |
| `cantidad` | Entre 1 y 10000 | **Log & Alert** | 🟡 MEDIUM | Cantidades extremas pueden ser válidas pero requieren revisión. |

### 🛠️ Herramientas de Contratos de Datos

#### **Pandas Validations** (Python - para prototipado)
```python
# Validaciones inline
assert df['venta_id'].notna().all(), "Hay ventas sin ID"
assert (df['monto'] > 0).all(), "Hay montos negativos"
```

### 📊 Ejemplo Real de Contrato Fallido

**Escenario**: Recibimos un CSV con fechas en formato mixto:
```
venta_id,fecha,monto
1,2024-01-15,150.50
2,15/01/2024,200.00    ← Formato inconsistente
3,2024-01-16,NULL      ← Monto nulo
```

**Decisión según contrato:**
1. Row 1: ✅ PASS → Va a Silver
2. Row 2: ⚠️ WARN → Auto-fix fecha, log warning, va a Silver
3. Row 3: ❌ FAIL → Va a `bronze_quarantine` para revisión manual

> [!TIP]
> En producción estos contratos se definen declarativamente (Pydantic / YAML) y se ejecutan en cada corrida del pipeline.

---

## 🔍 2. Data Quality Checks Automáticos

Antes de comenzar con las transformaciones, ejecutamos **checks de calidad** sobre los datos Bronze para detectar problemas temprano.

### 📈 Métricas de Calidad (6 Dimensiones)

| Dimensión | Descripción | Ejemplo de Check | Umbral |
| :--- | :--- | :--- | :--- |
| **Completitud** | ¿Cuántos campos están completos? | `COUNT(cliente_id) / COUNT(*)` | ≥ 95% |
| **Unicidad** | ¿Hay duplicados? | `COUNT(DISTINCT venta_id) = COUNT(*)` | 100% |
| **Validez** | ¿Los valores están en rangos esperados? | `fecha BETWEEN '2020-01-01' AND NOW()` | 100% |
| **Consistencia** | ¿Los datos concuerdan entre tablas? | `cliente_id` existe en `dim_clientes` | ≥ 98% |
| **Precisión** | ¿Los valores son correctos? | `total = precio * cantidad` | ≥ 99% |
| **Frescura** | ¿Cuán recientes son los datos? | `MAX(fecha_ingesta) > NOW() - INTERVAL '1 day'` | < 24h |

### 📊 Dashboard de Calidad (Ejemplo Output)

```
=== QUALITY CHECK RESULTS ===
Check                      | Status | Score  | Threshold
---------------------------|--------|--------|----------
Completitud Cliente        | ✅ PASS | 98.5%  | ≥ 95%
Unicidad Venta ID          | ✅ PASS | 100%   | 100%
Validez Montos             | ⚠️ WARN | 97.2%  | ≥ 99%
Consistencia Cliente FK    | ✅ PASS | 99.1%  | ≥ 98%
Frescura Datos             | ✅ PASS | 2h ago | < 24h

OVERALL STATUS: ⚠️ WARNING - 1 check below threshold
ACTION REQUIRED: Review 2.8% of records with invalid montos
```

### 🚨 Acciones Automáticas Basadas en Calidad

```python
def evaluar_calidad_y_decidir(df, quality_score):
    """
    Decide qué hacer basado en el score de calidad
    """
    if quality_score >= 95:
        # Calidad excelente - procesar normalmente
        return "PROCESS_TO_SILVER"
    elif quality_score >= 85:
        # Calidad aceptable - procesar con warning
        logger.warning(f"Quality score {quality_score}% - Processing with caution")
        return "PROCESS_WITH_WARNING"
    else:
        # Calidad inaceptable - detener pipeline
        logger.error(f"Quality score {quality_score}% - Pipeline halted")
        return "HALT_AND_ALERT"
```

---

In [7]:
# Simulamos datos Bronze con problemas tipicos
data_bronze = {
    'venta_id': [1, 2, 3, 4, 5, 6, 7, 8],
    'producto': ['  Laptop  ', 'MOUSE', 'Teclado', 'Monitor', None, 'laptop', 'Mouse', 'Auriculares'],
    'precio': [1500.50, 25.00, 75.00, 350.00, 120.00, 1450.00, None, 85.50],
    'cantidad': [1, 2, None, 1, 3, 1, 5, 2],
    'fecha': ['2024-01-15', '15/01/2024', '2024-01-17', '2024-01-18', '2024-01-19', 'invalid', '2024-01-21', '2024-01-22'],
    'cliente_email': ['juan@example.com', 'MARIA@EXAMPLE.COM', 'pedro@example.com', None, 'ana@example.com', 'luis@example.com', 'sofia@example.com', 'carlos@EXAMPLE.COM']
}

df_bronze = pd.DataFrame(data_bronze)
print("=== DATOS BRONZE (Con problemas) ===")
print(df_bronze)
print("\n" + "="*50 + "\n")


=== DATOS BRONZE (Con problemas) ===
   venta_id     producto  precio  cantidad       fecha       cliente_email
0         1     Laptop    1500.5       1.0  2024-01-15    juan@example.com
1         2        MOUSE    25.0       2.0  15/01/2024   MARIA@EXAMPLE.COM
2         3      Teclado    75.0       NaN  2024-01-17   pedro@example.com
3         4      Monitor   350.0       1.0  2024-01-18                None
4         5         None   120.0       3.0  2024-01-19     ana@example.com
5         6       laptop  1450.0       1.0     invalid    luis@example.com
6         7        Mouse     NaN       5.0  2024-01-21   sofia@example.com
7         8  Auriculares    85.5       2.0  2024-01-22  carlos@EXAMPLE.COM




In [8]:
# === FIXING: Limpieza de datos ===

# 1. Normalizar strings (strip, lower, remover espacios extra)
df_bronze['producto'] = df_bronze['producto'].str.strip().str.lower().str.title()
df_bronze['cliente_email'] = df_bronze['cliente_email'].str.strip().str.lower()

# 2. Manejar valores nulos segun regla de negocio
# - cantidad NULL -> 1 (asumimos cantidad minima)
# - precio NULL -> marcar para revision
# - producto NULL -> 'Desconocido'
df_bronze['cantidad'] = df_bronze['cantidad'].fillna(1)
df_bronze['producto'] = df_bronze['producto'].fillna('Desconocido')

# 3. Estandarizar fechas (parsear multiples formatos)
from dateutil import parser

def parse_fecha_flexible(fecha_str):
    """Intenta parsear fecha en multiples formatos"""
    try:
        if pd.isna(fecha_str):
            return None
        return parser.parse(str(fecha_str)).strftime('%Y-%m-%d')
    except:
        return None

df_bronze['fecha_fix'] = df_bronze['fecha'].apply(parse_fecha_flexible)

# 4. Crear columnas de metadatos de calidad
df_bronze['calidad_precio'] = df_bronze['precio'].notna()
df_bronze['calidad_fecha'] = df_bronze['fecha_fix'].notna()
df_bronze['registro_valido'] = df_bronze['calidad_precio'] & df_bronze['calidad_fecha']

# 5. Separar registros validos vs cuarentena
df_silver_ready = df_bronze[df_bronze['registro_valido']].copy()
df_quarantine = df_bronze[~df_bronze['registro_valido']].copy()

print("=== DATOS SILVER (Limpios) ===")
print(df_silver_ready[['venta_id', 'producto', 'precio', 'cantidad', 'fecha_fix', 'cliente_email']])
print(f"\nTotal registros validos: {len(df_silver_ready)}")


=== DATOS SILVER (Limpios) ===
   venta_id     producto  precio  cantidad   fecha_fix       cliente_email
0         1       Laptop  1500.5       1.0  2024-01-15    juan@example.com
1         2        Mouse    25.0       2.0  2024-01-15   maria@example.com
2         3      Teclado    75.0       1.0  2024-01-17   pedro@example.com
3         4      Monitor   350.0       1.0  2024-01-18                None
4         5  Desconocido   120.0       3.0  2024-01-19     ana@example.com
7         8  Auriculares    85.5       2.0  2024-01-22  carlos@example.com

Total registros validos: 6


In [9]:
print("\n=== DATOS EN CUARENTENA (Para revision manual) ===")
print(df_quarantine[['venta_id', 'producto', 'precio', 'fecha', 'fecha_fix']])
print(f"Total registros en cuarentena: {len(df_quarantine)}")



=== DATOS EN CUARENTENA (Para revision manual) ===
   venta_id producto  precio       fecha   fecha_fix
5         6   Laptop  1450.0     invalid        None
6         7    Mouse     NaN  2024-01-21  2024-01-21
Total registros en cuarentena: 2


In [10]:
# === METRICAS DE LIMPIEZA ===
print("\n=== METRICAS DE CALIDAD POST-LIMPIEZA ===")
print(f"Tasa de exito: {len(df_silver_ready)/len(df_bronze)*100:.1f}%")
print(f"Fechas estandarizadas: {df_bronze['fecha_fix'].notna().sum()}/{len(df_bronze)}")
print(f"Emails normalizados: {df_bronze['cliente_email'].notna().sum()}/{len(df_bronze)}")
print("\nLimpieza tecnica completada")


=== METRICAS DE CALIDAD POST-LIMPIEZA ===
Tasa de exito: 75.0%
Fechas estandarizadas: 7/8
Emails normalizados: 7/8

Limpieza tecnica completada


### 🔀 ¿Y enriquecer / agrupar / segmentar? → eso es Gold (clase05)

Silver deja el dato **confiable a nivel registro**: normalizaciones, flags de calidad, parseo de tipos, categorías homologadas, rango etario y campos derivados básicos — siempre **mismo grano** (una fila por venta) y de **uso general**.

Cuando esos atributos se **agregan** (`GROUP BY`), se arman **segmentos o métricas finales**, se hace **JOIN con dimensiones** o se construye un **producto analítico (dashboard / ABT para ML)** → ya **no es Silver, es Gold**. El Star Schema, los pivots, la segmentación de negocio y la ABT se hacen en **[clase05 (Gold)](../clase05/clase05.ipynb)**.

> Regla práctica: **Silver = el dato correcto, fila por fila. Gold = el dato modelado/agregado para una pregunta de negocio.**


## 🕰️ 3. Historización (SCD) y Normalización

**¿Qué pasa si un atributo cambia en el origen?** (un cliente se muda, un producto se renombra). En vez de pisar el dato, hay técnicas para **no perder la historia**:

- **SCD2 (histórico completo):** por cada cambio insertás una **fila nueva** con `fecha_inicio` / `fecha_fin` / `version_activa` y cerrás la anterior. Mantenés todas las versiones (auditoría, "como era en su momento").
- **SCD3 (versión anterior):** agregás una columna extra (`valor_anterior`) — sólo guardás el penúltimo valor. Uso limitado (cuando alcanza con "antes vs ahora").

| | SCD1 (sobrescribir) | SCD2 (fila nueva) | SCD3 (columna extra) |
|---|---|---|---|
| Historia | ❌ ninguna | ✅ completa | ⚠️ sólo la previa |
| Cuándo | corrección de typo | atributo de negocio que cambia | comparación simple antes/ahora |

> En este curso la historización profunda (dimensiones SCD2 en el modelo) se ve en **Gold (clase05)**; en Silver alcanza con conocer el patrón y dejar el dato listo.

---

### 🏗️ Normalización Técnica en Silver

**Normalizar** = guardar cada entidad en **su propia tabla, sin repetir datos**, vinculadas por **claves foráneas (FK)**. La **3ª Forma Normal (3NF)** es el nivel donde una tabla no tiene columnas derivadas ni dependencias transitivas: cada dato vive en **un solo lugar**. En Silver conviene porque mantiene **integridad referencial** y permite **actualizar en un solo lugar** (no se desincroniza). Gold después *desnormaliza* (Star Schema) para que el negocio consulte fácil.

En Silver, intentamos mantener las entidades **separadas y normalizadas** (3NF):

```
silver_ventas (Transacciones)
├── venta_id (PK)
├── cliente_id (FK → silver_clientes)
├── producto_id (FK → silver_productos)
├── sucursal_id (FK → silver_sucursales)
└── fecha, monto, cantidad

silver_clientes (Maestro de personas - SCD Tipo 2)
├── pk (Surrogate Key)
├── cliente_id (Business Key)
├── nombre, email, ciudad
├── fecha_inicio, fecha_fin
└── version_activa

silver_productos (Maestro de artículos - SCD Tipo 1)
├── producto_id (PK)
├── nombre, categoria
└── precio_actual

silver_sucursales (Maestro de locales - SCD Tipo 2)
├── pk (Surrogate Key)
├── sucursal_id (Business Key)
├── nombre, ciudad, region
├── fecha_apertura, fecha_cierre
└── activa
```

### 🎯 Regla de Oro Silver vs Gold

| Capa | Audiencia | Estrategia | Ejemplo |
|:---|:---|:---|:---|
| **Silver** | Ingenieros / Data Scientists | **Normalizado (3NF)** | Tablas separadas con FKs |
| **Gold** | Analistas / BI | **Desnormalizado (Star)** | Tablas anchas con JOINs pre-hechos |

> **¿Por qué normalizar en Silver?**
> - Ahorra espacio (no repetimos datos)
> - Mantiene integridad referencial
> - Facilita updates (cambio en 1 lugar)
> - Permite SCD sin duplicar todo

> **¿Por qué desnormalizar en Gold?**
> - Queries más simples para negocio
> - Performance en lecturas (no JOINs)
> - Power BI/Tableau prefieren tablas anchas

---

## 🚀 4. SQL Pushdown (ELT)

En este notebook hicimos la limpieza con **pandas** porque es **didáctico**: se ve fila por fila, en memoria. Pero en **producción** la transformación Bronze→Silver normalmente **no** se hace trayendo los datos a un worker de Python — se hace con **SQL ejecutado *dentro* de la base** (patrón **ELT / "pushdown"**: empujás el cómputo a donde están los datos). La celda de abajo (`sql_silver`) es exactamente eso: la misma limpieza, en una sentencia SQL.

**¿Por qué SQL en lugar de pandas en producción?**

- **Escala**: no traés millones de filas a la RAM del worker; el dato se procesa donde vive.
- **El motor optimiza**: el planner de la DB usa índices, paraleliza y elige el mejor plan.
- **Menos transferencia**: viaja el resultado, no toda la tabla por la red.
- **Set-based**: un `INSERT … SELECT` / `UPDATE` en vez de un loop fila por fila.
- **Idempotente**: `CREATE OR REPLACE TABLE …` re-corre sin duplicar (igual que el `if_exists="replace"` que usan los DAGs).
- **Versionable**: el SQL queda en archivos / dbt, lo lee y revisa todo el equipo.

> **Regla práctica:** *pandas para aprender y prototipar; SQL pushdown para producción a escala.* Los DAGs de la sección 6 cargan con `pandas.to_sql` / `if_exists="replace"` por simplicidad didáctica — el patrón productivo equivalente es este SQL.


In [11]:
sql_silver = """
CREATE OR REPLACE TABLE silver_ventas AS
SELECT 
    venta_id,
    UPPER(TRIM(producto)) as producto,
    COALESCE(cantidad, 0) as cantidad
FROM bronze_ventas
WHERE precio > 0;
"""
print("✅ Transformación via SQL ejecutada")

✅ Transformación via SQL ejecutada


## 🎓 5. Best Practices para Capa Silver

### ✅ DO - Sí hacer

| Práctica | Razón | Ejemplo |
|:---|:---|:---|
| **Usar tipos de datos correctos** | Ahorra espacio y permite operaciones nativas | `fecha` como DATE, no STRING |
| **Agregar metadatos de calidad** | Permite auditoría y debugging | `_ingesta_timestamp`, `_calidad_score` |
| **Implementar idempotencia** | Re-runs no duplican datos | `MERGE` o `INSERT ON CONFLICT` |
| **Documentar transformaciones** | El siguiente ingeniero agradecerá | Docstrings + comments en lógica compleja |
| **Testear contratos automáticamente** | Detecta errores antes de producción | tests de contrato (Pydantic) en CI/CD |
| **Separar datos válidos de cuarentena** | No mezclar datos limpios con sucios | `silver_ventas` vs `quarantine_ventas` |

### ❌ DON'T - No hacer

| Anti-Práctica | Problema | Mejor Alternativa |
|:---|:---|:---|
| **Sobrescribir sin SCD cuando hay historia** | Pierdes auditoría | Implementar SCD Tipo 2 |
| **Dejar NULLs sin manejar** | Queries explotan o dan resultados incorrectos | `COALESCE`, `fillna`, o quarantine |
| **Hardcodear valores en código** | Cambios requieren redeploy | Config files o tablas de parámetros |
| **Ignorar duplicados** | Métricas infladas incorrectamente | `DISTINCT` o validación de unicidad |
| **Joins sin validar FK** | Datos huérfanos no detectados | `LEFT JOIN` + contar NULLs |
| **Transformaciones en una línea gigante** | Imposible debuggear | Pasos intermedios con nombres claros |

### 🏗️ Patrón de Naming Convenciones

```
Capa Bronze: bronze_<source>_<entity>
  Ejemplo: bronze_api_ventas, bronze_csv_clientes

Capa Silver: silver_<entity> o silver_<domain>_<entity>
  Ejemplo: silver_ventas, silver_crm_clientes

Tablas Especiales:
  - quarantine_<entity>      → Datos que fallaron validación
  - staging_<entity>          → Datos temporales durante transformación
  - audit_<entity>            → Log de cambios (SCD histórico)
```


---

## 🐳 6. Pipeline Silver con Airflow — DAGs Pedagógicos

Hasta acá vimos los conceptos en notebooks. Ahora **bajamos esos conceptos a un DAG real** que vas a poder correr en Airflow.

Dos DAGs incrementales (mismo patrón que `bronze_01..04` de clase03: `dag_id` == nombre del archivo):

| DAG (`dag_id` == archivo) | Genera | Concepto |
|---|---|---|
| `silver_01_basico.py` | `silver.ventas_basico` | Limpieza básica: strip + Title Case + fillna + parser flexible de fechas (sin validar, sin quarantine) |
| `silver_02_contrato.py` | `silver.ventas_contrato` + `silver.quarantine_ventas_contrato` | Contrato **contract-driven** (Pydantic generado desde `ventas.yaml`) + Pattern Quarantine + Audit |

Cada cell `%%writefile` escribe un DAG en `stack/dags/02-silver/` al ejecutarse; aparece en Airflow UI (`localhost:8080`).

### 📊 Cómo quedan las tablas (corré los 2 DAGs y comparalos)

Cada DAG escribe a **tablas distintas**, así que podés correr ambos y ver el contraste:

| Tabla | La crea | Qué tiene |
|---|---|---|
| `silver.ventas_basico` | `silver_01_basico` | **8 filas** — todo cargado, limpio pero **sin validar** (no separa inválidos) |
| `silver.ventas_contrato` | `silver_02_contrato` | sólo las filas **válidas** según `ventas.yaml` (+ `silver_at`) |
| `silver.quarantine_ventas_contrato` | `silver_02_contrato` | las **rechazadas**, con `error_type` / `error_message` / `quarantined_at` |

```sql
SELECT 'ventas_basico'               AS tabla, count(*) FROM silver.ventas_basico
UNION ALL SELECT 'ventas_contrato',             count(*) FROM silver.ventas_contrato
UNION ALL SELECT 'quarantine_ventas_contrato',  count(*) FROM silver.quarantine_ventas_contrato;
```

`ventas_basico` mete **todo** (8); `ventas_contrato` + `quarantine_ventas_contrato` **suman 8** pero separados → ahí se ve el valor del contrato.

---

### 🐳 DAG 1: Silver Básico

Toma datos crudos sintéticos en `bronze.ventas_demo` (los crea el propio DAG), aplica limpieza básica (strip, fillna, parser flexible de fechas) y carga `silver.ventas_basico`.


### 📋 Runbook: `silver_01_basico`

> **🎯 Qué hace**: lee `bronze.ventas_demo` (datos crudos sintéticos creados por la primera task del DAG), aplica limpieza básica (strip + Title Case + fillna + parser flexible de fechas), escribe `silver.ventas_basico` con `if_exists="replace"`.
>
> **Pasos para correrlo**:
>
> 1. **Pre-requisito**: ninguno — el DAG crea `bronze.ventas_demo` desde cero (task `prepare_bronze`). NO necesitás haber corrido nada de clase 03.
> 2. **Ejecutar la celda de abajo** → escribe `silver_01_basico.py` en `stack/dags/02-silver/`.
> 3. **En Airflow UI** (`localhost:8080`):
>    - Filtrá por tag `silver`
>    - Buscá `silver_01_basico` (su `dag_id`, igual que el nombre del archivo)
>    - Activá el toggle si está pausado y click ▶️ "Trigger DAG"
> 4. **Verificar en Postgres** (DBeaver o `psql`):
>    ```sql
>    SELECT * FROM silver.ventas_basico LIMIT 10;
>    ```
> 5. **Estado esperado post-corrida**:
>    - 8 filas en `silver.ventas_basico`
>    - Sin `quarantine` (este DAG no separa inválidos — los limpia y carga todos)
>    - **Idempotente**: re-correr el DAG NO duplica — `if_exists="replace"` reescribe la tabla (resultado determinístico).
>
> **👀 Qué observar específicamente**:
> - **Strings normalizados**: `"  Laptop  "` → `"Laptop"`, `"MOUSE"` → `"Mouse"`, emails en lowercase.
> - **Fechas parseadas flexible**: `"15/01/2024"` (formato DD/MM/YYYY) y `"2024-01-15"` (ISO) ambas se convierten a `date` objeto.
> - **`cantidad` con nulos** → fillna(1) los reemplaza por 1.
> - **NO valida calidad**: aunque `producto = None` y `cantidad = None`, los acepta silenciosamente. Eso es lo que el siguiente DAG (`silver_02_contrato`) viene a arreglar.

In [12]:
%%writefile ../stack/dags/02-silver/silver_01_basico.py

"""
DAG pedagogico: Silver Basico
Clase 04 - Limpieza basica de datos crudos a Silver.

Pipeline didactico (datos sinteticos):
  bronze.ventas_demo (sucio) -> normalizar + fillna + parse fechas -> silver.ventas_basico
"""

from airflow.decorators import dag, task
from datetime import datetime
import os


DB_URI = (
    f"postgresql+psycopg2://"
    f"{os.getenv('SOURCE_DB_USER', 'admin')}:"
    f"{os.getenv('SOURCE_DB_PASS', 'admin')}@"
    f"{os.getenv('SOURCE_DB_HOST', 'data_warehouse')}:5432/"
    f"{os.getenv('SOURCE_DB_NAME', 'InfraCienciaDatos')}"
)


@dag(
    dag_id="silver_01_basico",
    start_date=datetime(2024, 1, 1),
    schedule=None,
    catchup=False,
    tags=["silver"],
    doc_md="DAG didactico: limpieza basica Bronze -> Silver con datos sinteticos.",
)
def silver_01_basico():

    @task
    def prepare_bronze():
        """Crea bronze.ventas_demo con datos crudos sucios (8 filas)."""
        import pandas as pd
        import sqlalchemy

        data = {
            "venta_id": [1, 2, 3, 4, 5, 6, 7, 8],
            "producto": ["  Laptop  ", "MOUSE", "Teclado", "Monitor",
                         None, "laptop", "Mouse", "Auriculares"],
            "precio": [1500.50, 25.00, 75.00, 350.00,
                       120.00, 1450.00, 50.00, 85.50],
            "cantidad": [1, 2, None, 1, 3, 1, 5, 2],
            "fecha": ["2024-01-15", "15/01/2024", "2024-01-17", "2024-01-18",
                      "2024-01-19", "2024-01-20", "2024-01-21", "2024-01-22"],
            "cliente_email": ["juan@example.com", "MARIA@EXAMPLE.COM",
                              "pedro@example.com", None, "ana@example.com",
                              "luis@example.com", "sofia@example.com",
                              "carlos@EXAMPLE.COM"],
        }
        df = pd.DataFrame(data)

        engine = sqlalchemy.create_engine(DB_URI)
        with engine.begin() as conn:
            conn.execute(sqlalchemy.text("CREATE SCHEMA IF NOT EXISTS bronze;"))
        df.to_sql("ventas_demo", engine, schema="bronze", if_exists="replace", index=False)  # idempotente: replace -> re-run no duplica
        print(f"bronze.ventas_demo creada con {len(df)} filas crudas.")

    @task
    def clean_silver():
        """Lee bronze, aplica limpieza basica, escribe silver."""
        import pandas as pd
        import sqlalchemy
        from dateutil import parser

        engine = sqlalchemy.create_engine(DB_URI)
        df = pd.read_sql("SELECT * FROM bronze.ventas_demo", engine)

        # 1. Strings: strip + Title Case + nulos -> "Desconocido"
        df["producto"] = df["producto"].fillna("Desconocido").str.strip().str.title()
        df["cliente_email"] = df["cliente_email"].fillna("").str.strip().str.lower()

        # 2. Cantidad: nulos -> 1, tipado entero
        df["cantidad"] = df["cantidad"].fillna(1).astype(int)

        # 3. Fechas: parser flexible (acepta "2024-01-15" y "15/01/2024")
        def parse_fecha(s):
            try:
                return parser.parse(str(s)).date()
            except Exception:
                return None
        df["fecha"] = df["fecha"].apply(parse_fecha)

        # 4. Cargar
        with engine.begin() as conn:
            conn.execute(sqlalchemy.text("CREATE SCHEMA IF NOT EXISTS silver;"))
        df.to_sql("ventas_basico", engine, schema="silver", if_exists="replace", index=False)  # idempotente: replace -> re-run no duplica (deterministico)
        print(f"silver.ventas_basico cargada con {len(df)} filas limpias.")

    prepare_bronze() >> clean_silver()


silver_01_basico()


Writing ../stack/dags/02-silver/silver_01_basico.py


### 🐳 DAG 2: Silver contract-driven (Pydantic desde `ventas.yaml`) + Quarantine

Mismo input que el DAG 1, pero ahora **validamos cada fila contra un contrato** y separamos las inválidas a `silver.quarantine_ventas_contrato` (en vez de descartarlas), con audit metadata (`silver_at`, `quarantined_at`).

**Hardcodear el contrato en el DAG es frágil** — si cambia una regla hay que tocar y redeployar el DAG:

```python
class VentaContract(BaseModel):           # ← frágil: vive enterrado en el DAG
    venta_id: int = Field(gt=0)
    producto: str = Field(min_length=2)
    precio:  float = Field(gt=0)
    cantidad: int = Field(default=1, ge=1)
    fecha: date
    cliente_email: Optional[EmailStr] = None
```

La versión profesional es **contract-driven**: el modelo Pydantic se construye en runtime desde el YAML que creamos en clase03 ([`stack/data/contracts/ventas.yaml`](../stack/data/contracts/ventas.yaml)) con `build_pydantic_from_contract()` ([`stack/dags/common/contracts.py`](../stack/dags/common/contracts.py)):

```python
from common.contracts import load_contract, build_pydantic_from_contract
contract = load_contract("/opt/airflow/data/contracts/ventas.yaml")
VentaContract = build_pydantic_from_contract(contract)   # generado en runtime
```

#### El círculo se cierra (un contrato, dos capas)

```mermaid
graph LR
    Y["ventas.yaml<br/>UN contrato"] --> B["Bronze (clase03)<br/>validate_file_shape()<br/>FORMA"]
    Y --> S["Silver (clase04)<br/>build_pydantic_from_contract()<br/>SEMANTICA fila por fila"]
    style Y fill:#fff9c4,stroke:#fbc02d
    style B fill:#e8f5e9,stroke:#2e7d32
    style S fill:#e3f2fd,stroke:#1565c0
```

> **📋 Runbook `silver_02_contrato`**: ejecutá la celda → escribe `silver_02_contrato.py`. En Airflow UI buscá `silver_02_contrato` (`dag_id` == archivo) → activar + trigger. Genera `silver.ventas_contrato` (válidas + `silver_at`) y `silver.quarantine_ventas_contrato` (rechazadas + `error_type`/`error_message`/`quarantined_at`). En los logs de `validate_with_contract` aparece `Contrato cargado: ventas vX` — confirma que el Pydantic se construyó desde el YAML. **Por qué cae cada fila a quarantine** (validado contra `ventas.yaml`):

| `venta_id` | Campo | Valor en Bronze | Regla del contrato | Por qué cae |
|---|---|---|---|---|
| 4 | `cliente_email` | `"no-es-email"` | `type: email` | no tiene formato de email válido (`EmailStr`) |
| 6 | `producto` | `" "` | `min_length: 2` | tras `trim` queda vacío / < 2 chars |
| 6 | `precio` | `-50.00` | `gt: 0` | el precio no es estrictamente positivo |
| 6 | `fecha` | `"invalid-date"` | `type: date` | no parsea como fecha |
| 7 | `precio` | `None` | `nullable: false` | campo requerido: no puede ser nulo |

El resto (1, 2, 3, 5, 8) cumple todas las reglas → va a `silver.ventas_contrato`. **Quarantine no descarta: aísla la fila con el motivo** (`error_type` / `error_message`) para auditarla o corregir el contrato.


In [13]:
%%writefile ../stack/dags/02-silver/silver_02_contrato.py

"""
DAG pedagogico: Silver con Pydantic CONTRACT-DRIVEN + Quarantine
Clase 04 - Validacion estricta usando el contrato YAML compartido con Bronze.

Pipeline:
  bronze.ventas_demo
    -> Pydantic generado dinamicamente desde ventas.yaml
    -> { silver.ventas_contrato | silver.quarantine_ventas_contrato }

Diferencia con la version anterior:
  - YA NO hay `class VentaContract(BaseModel)` hardcodeado.
  - El contrato se LEE de stack/data/contracts/ventas.yaml.
  - Pydantic se construye en runtime con build_pydantic_from_contract().
  - Cambiar el contrato (agregar columnas, ajustar reglas) = editar YAML.
"""

from airflow.decorators import dag, task
from datetime import datetime
import os
import sys

# Hacemos visible el paquete `common` (esta dos niveles arriba)
sys.path.append("/opt/airflow/dags")
from common.contracts import build_pydantic_from_contract, load_contract  # noqa: E402


DB_URI = (
    f"postgresql+psycopg2://"
    f"{os.getenv('SOURCE_DB_USER', 'admin')}:"
    f"{os.getenv('SOURCE_DB_PASS', 'admin')}@"
    f"{os.getenv('SOURCE_DB_HOST', 'data_warehouse')}:5432/"
    f"{os.getenv('SOURCE_DB_NAME', 'InfraCienciaDatos')}"
)

CONTRACT_PATH = "/opt/airflow/data/contracts/ventas.yaml"


@dag(
    dag_id="silver_02_contrato",
    start_date=datetime(2024, 1, 1),
    schedule=None,
    catchup=False,
    tags=["silver"],
    doc_md="DAG didactico: Pydantic generado dinamicamente desde ventas.yaml + quarantine.",
)
def silver_02_contrato():

    @task
    def prepare_bronze():
        """Crea bronze.ventas_demo con datos sucios (incluye filas invalidas)."""
        import pandas as pd
        import sqlalchemy

        # 8 filas: 3 fallan el contrato ventas.yaml (van a quarantine):
        #   venta_id=4 -> cliente_email "no-es-email": type:email invalido
        #   venta_id=6 -> producto " " (min_length:2) + precio -50 (gt:0)
        #                 + fecha "invalid-date" (type:date)  [3 violaciones]
        #   venta_id=7 -> precio None: el contrato lo exige (nullable:false)
        # Las otras 5 (1,2,3,5,8) pasan -> silver.ventas_contrato
        data = {
            "venta_id": [1, 2, 3, 4, 5, 6, 7, 8],
            "producto": ["Laptop", "Mouse", "Teclado", "Monitor",
                         "Auriculares", " ", "Mouse", "Tablet"],
            "precio": [1500.50, 25.00, 75.00, 350.00,
                       120.00, -50.00, None, 85.50],
            "cantidad": [1, 2, 1, 1, 3, 1, 5, 2],
            "fecha": ["2024-01-15", "2024-01-16", "2024-01-17", "2024-01-18",
                      "2024-01-19", "invalid-date", "2024-01-21", "2024-01-22"],
            "cliente_email": ["juan@example.com", "maria@example.com",
                              "pedro@example.com", "no-es-email",
                              "ana@example.com", "luis@example.com",
                              "sofia@example.com", "carlos@example.com"],
        }
        df = pd.DataFrame(data)

        engine = sqlalchemy.create_engine(DB_URI)
        with engine.begin() as conn:
            conn.execute(sqlalchemy.text("CREATE SCHEMA IF NOT EXISTS bronze;"))
        df.to_sql("ventas_demo", engine, schema="bronze", if_exists="replace", index=False)  # idempotente: replace -> re-run no duplica
        print(f"bronze.ventas_demo: {len(df)} filas (algunas invalidas).")

    @task
    def validate_with_contract():
        """Lee contrato YAML, construye Pydantic, valida fila por fila."""
        import pandas as pd
        import sqlalchemy

        # 1) Leer contrato Y construir Pydantic en runtime (corazon del refactor)
        contract = load_contract(CONTRACT_PATH)
        VentaContract = build_pydantic_from_contract(contract)
        print(f"Contrato cargado: {contract['dataset']} v{contract['version']}")
        print(f"Modelo Pydantic generado con campos: {list(VentaContract.model_fields.keys())}")

        # 2) Leer Bronze
        engine = sqlalchemy.create_engine(DB_URI)
        df = pd.read_sql("SELECT * FROM bronze.ventas_demo", engine)

        # 3) Validar fila por fila
        validos, invalidos = [], []
        for _, row in df.iterrows():
            try:
                ok = VentaContract(**row.to_dict())
                d = ok.model_dump()
                # serializar fechas a ISO para la DB
                if d.get("fecha"):
                    d["fecha"] = d["fecha"].isoformat()
                validos.append(d)
            except Exception as e:
                inv = {k: (v if not pd.isna(v) else None) for k, v in row.to_dict().items()}
                inv["error_type"] = type(e).__name__
                inv["error_message"] = str(e)[:300]
                invalidos.append(inv)

        print(f"Validos: {len(validos)} | Invalidos: {len(invalidos)}")
        return {"validos": validos, "invalidos": invalidos}

    @task
    def load_silver(payload: dict):
        """Carga validos en silver.ventas_contrato + audit metadata."""
        import pandas as pd
        import sqlalchemy
        from datetime import datetime as dt

        engine = sqlalchemy.create_engine(DB_URI)
        df = pd.DataFrame(payload["validos"])
        if df.empty:
            print("Sin validos para silver.")
            return
        df["silver_at"] = dt.now().isoformat()

        with engine.begin() as conn:
            conn.execute(sqlalchemy.text("CREATE SCHEMA IF NOT EXISTS silver;"))
        df.to_sql("ventas_contrato", engine, schema="silver", if_exists="replace", index=False)  # idempotente: replace -> re-run no duplica
        print(f"silver.ventas_contrato: {len(df)} filas validas.")

    @task
    def load_quarantine(payload: dict):
        """Carga invalidos en silver.quarantine_ventas_contrato + error info."""
        import pandas as pd
        import sqlalchemy
        from datetime import datetime as dt

        engine = sqlalchemy.create_engine(DB_URI)
        df = pd.DataFrame(payload["invalidos"])
        if df.empty:
            print("Sin invalidos en cuarentena.")
            return
        df["quarantined_at"] = dt.now().isoformat()

        with engine.begin() as conn:
            conn.execute(sqlalchemy.text("CREATE SCHEMA IF NOT EXISTS silver;"))
        df.to_sql("quarantine_ventas_contrato", engine, schema="silver", if_exists="replace", index=False)  # idempotente: replace -> re-run no duplica
        print(f"silver.quarantine_ventas_contrato: {len(df)} filas invalidas.")

    bronze_done = prepare_bronze()
    payload = validate_with_contract()
    bronze_done >> payload
    load_silver(payload)
    load_quarantine(payload)


silver_02_contrato()


Writing ../stack/dags/02-silver/silver_02_contrato.py


---
## ⏭️ ¿Y ahora qué?

> [!IMPORTANT]
> Hemos convertido el "barro" de Bronze en "acero" de Silver: limpio, tipado, normalizado y con control de historia (SCD).
>
> El siguiente paso es la **Clase 05 (Gold Layer)**, donde uniremos estas piezas en un Star Schema y tablas anchas para ML.

###  ¡Capa Silver: Refinería Terminada!\n\nHas aprendido a transformar datos crudos en información confiable mediante contratos de calidad y gestión de cuarentena.\n\n **Práctica entregable**: el ejercicio de esta clase es [`ejercicios/ejercicio.ipynb`](ejercicios/ejercicio.ipynb) — Setup de Northwind (con datos sucios sembrados a propósito) + 8 ejercicios de SQL básico (filtros, COUNT/MIN/MAX/AVG, normalización+nulos, atributo derivado con CASE, un JOIN), los fundamentos que usa Silver a nivel de registro. Ahí generás tu entrega.